# Unit 7 solutions: plan the lap, then find its edge

These reference solutions build the speed profile from its two passes, predict each answer first where the lesson gives a law (the braking ramp, the lead, the friction circle), and then measure it.
Every race is seeded, so every number below is reproducible, and executable asserts check each answer against an independent derivation.
Every number is a number about racekit's **carla-like car**: a model, not the truth.

In [ ]:
import math
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import carla_like
from racekit.measure import noise_aware_better, repeat_races
from racekit.runner import load_submission, run_race, shim_installed
from racekit.tracks import grip_sections, load_track

AGENTS = Path("../../projects/racekit/agents")
UNIT_06, UNIT_07 = AGENTS / "unit-06.py", AGENTS / "unit-07.py"
workdir = Path(mkdtemp(prefix="roar-unit-07-sol-"))
CLEAN_OFFSET = 2.0  # m: the clean rule's limit on the max |cross-track error|
CONTROL_DT = 0.05  # s: one telemetry row per control tick
TICK = 0.05 + 1e-9  # race times are pinned to one control tick


def variant(source, name, **constants):
    # Unit 1's idiom: a copy of a submission with some module constants changed, written into workdir.
    return write_variant(source, workdir / f"{name}.py", {key: repr(value) for key, value in constants.items()})


def max_offset(result):
    # The largest |cross-track error| over the driving rows (held == 0).
    telemetry = result.telemetry
    return float(np.max(np.abs(telemetry["offset"][telemetry["held"] == 0])))


def clean(result):
    # The unit's clean rule: finished, 0 collisions, and max |cross-track error| at most 2 m.
    return result.finished and result.collisions == 0 and max_offset(result) <= CLEAN_OFFSET

## Exercise 1 - The two passes on a straight

**Prediction.** The backward pass spreads the corner's 15 m/s back along the straight: k metres before waypoint 300 the profile is √(15² + 2·5·k), which reaches 45 m/s at k = (45² − 15²) / (2·5) = 180 m, so the ramp starts at waypoint 300 − 180 = **120**.
The forward pass climbs out of the corner as √(15² + 2·4·k) from waypoint 320, back to 45 m/s after (45² − 15²) / (2·4) = 225 m, at waypoint **545**.
Both squares are exact (√2025 = 45), so the ends of the ramps land exactly on whole waypoints.
The cap's own "lap" drops from 45 to 15 m/s inside one 1 m segment, a deceleration of (45² − 15²) / 2 = 900 m/s², about 92 g.

In [ ]:
def backward_pass(cap, ds, a_dec):
    v = list(cap)
    for i in range(len(v) - 2, -1, -1):
        v[i] = min(v[i], math.sqrt(v[i + 1] ** 2 + 2.0 * a_dec * ds[i]))
    return v


def forward_pass(v, ds, a_acc):
    v = list(v)
    for i in range(len(v) - 1):
        v[i + 1] = min(v[i + 1], math.sqrt(v[i] ** 2 + 2.0 * a_acc * ds[i]))
    return v


def straight_time(v, ds):
    # the sum over the segments of ds / (mean speed of its two ends)
    return sum(ds[i] / ((v[i] + v[i + 1]) / 2.0) for i in range(len(ds)))


toy_ds = [1.0] * 600
toy_cap = [15.0 if 300 <= i <= 320 else 45.0 for i in range(601)]
toy_profile = forward_pass(backward_pass(toy_cap, toy_ds, 5.0), toy_ds, 4.0)
ramp_start = min(i for i in range(600) if toy_profile[i + 1] < toy_profile[i] - 1e-9)
ramp_formula = (45.0 ** 2 - 15.0 ** 2) / (2.0 * 5.0)
back_to_top = min(i for i in range(321, 601) if toy_profile[i] >= 45.0 - 1e-9)
planned_time = straight_time(toy_profile, toy_ds)
cap_time = straight_time(toy_cap, toy_ds)
why_not_the_cap = (
    "The cap drops from 45 m/s to 15 m/s inside a single 1 m segment, which needs a deceleration of "
    "(45^2 - 15^2) / 2 = 900 m/s^2 (about 92 g), so no car can drive it: the cap is a limit for each "
    "waypoint, not a speed plan."
)

print(f"ramp starts at waypoint {ramp_start} ({300 - ramp_start} m before the corner; formula {ramp_formula:.1f} m)")
print(f"back to 45 m/s at waypoint {back_to_top} ({back_to_top - 320} m after the corner)")
print(f"planned time {planned_time:.4f} s, the cap's 'time' {cap_time:.4f} s")
print(why_not_the_cap)

# independent checks: the closed forms of the two ramps
assert len(toy_cap) == 601 and len(toy_profile) == 601
assert ramp_formula == 180.0 and ramp_start == 300 - 180 and back_to_top == 320 + 225
for k in (1, 50, 179):
    assert abs(toy_profile[300 - k] - math.sqrt(225.0 + 10.0 * k)) < 1e-9
    assert abs(toy_profile[320 + k] - math.sqrt(225.0 + 8.0 * k)) < 1e-9
assert toy_profile[300:321] == [15.0] * 21 and toy_profile[0] == 45.0 and toy_profile[600] == 45.0
# the time of a constant-acceleration ramp is (v_end - v_start) / a: 6 s braking, 7.5 s accelerating
ramp_time = (45.0 - 15.0) / 5.0 + (45.0 - 15.0) / 4.0
expected = ramp_time + 20.0 / 15.0 + (600.0 - 180.0 - 20.0 - 225.0) / 45.0
assert abs(planned_time - expected) < 0.01
assert abs(cap_time - (20.0 / 15.0 + 578.0 / 45.0 + 2.0 / 30.0)) < 1e-9
assert planned_time > cap_time

## Exercise 2 - The whole lap, on the oval

**Prediction.** `lap_profile` is the lesson's three steps with the cap built inside, so on the oval it should match the agent's own profile to rounding error (the agent ships every `GRIP` knob at 1.0).
The second lap of the passes only matters near the start line, where the backward pass reaches waypoint n − 1 before it has lowered waypoint 0, and the forward pass reaches waypoint 0 before it knows how fast the car leaves the last corner: on the oval that is a stretch of a straight on either side of the start: a tenth of the lap or so.
The slowest point is the apex of one of the two ends: the oval's ends are equal semicircles, so their caps are equal and the first of them wins the tie.

In [ ]:
def lap_profile(xs, ys, a_lat, a_dec, a_acc, top_speed=45.0, laps=2):
    n = len(xs)
    before, after = (np.arange(n) - 1) % n, (np.arange(n) + 1) % n
    twice_area = (xs - xs[before]) * (ys[after] - ys[before]) - (ys - ys[before]) * (xs[after] - xs[before])
    sides = (np.hypot(xs - xs[before], ys - ys[before]) * np.hypot(xs[after] - xs, ys[after] - ys)
             * np.hypot(xs[after] - xs[before], ys[after] - ys[before]))
    kappa = 2.0 * twice_area / sides  # Unit 2's Menger curvature, cyclic
    ds = np.hypot(xs[after] - xs, ys[after] - ys)  # segment i -> i + 1 (the last one closes the lap)
    v = [min(top_speed, math.sqrt(a_lat / abs(k))) if abs(k) > 1e-9 else top_speed for k in kappa]
    for _ in range(laps):
        for i in range(n - 1, -1, -1):
            v[i] = min(v[i], math.sqrt(v[(i + 1) % n] ** 2 + 2.0 * a_dec * ds[i]))
    for _ in range(laps):
        for i in range(n):
            v[(i + 1) % n] = min(v[(i + 1) % n], math.sqrt(v[i] ** 2 + 2.0 * a_acc * ds[i]))
    return np.array(v)


def oval_cap(xs, ys, a_lat, top_speed=45.0):
    # the cap alone: lap_profile with no passes at all
    return lap_profile(xs, ys, a_lat, 0.0, 0.0, top_speed, laps=0)


track = load_track("oval")
xs, ys, n = track.x, track.y, len(track.x)
after = (np.arange(n) + 1) % n  # the next waypoint, cyclic
ds_oval = np.hypot(xs[after] - xs, ys[after] - ys)
oval_profile = lap_profile(xs, ys, 4.6, 5.0, 4.0)
with shim_installed():
    Unit07, _ = load_submission(UNIT_07)
agent = Unit07(track.waypoints, None)  # agent.profile is its list of planned speeds
largest_difference = float(np.max(np.abs(oval_profile - np.array(agent.profile))))
changed_by_second_lap = int(np.sum(np.abs(lap_profile(xs, ys, 4.6, 5.0, 4.0, laps=1) - oval_profile) > 1e-9))
slowest_index = int(np.argmin(oval_profile))
slowest_speed = float(oval_profile[slowest_index])
planned_lap_time = float(np.sum(ds_oval / ((oval_profile + oval_profile[after]) / 2.0)))
cap = oval_cap(xs, ys, 4.6)
fraction_at_cap = float(np.mean(np.abs(oval_profile - cap) <= 1e-9))

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(np.arange(n), cap, color="lightgrey", label="the cap, sqrt(4.6 / |kappa|), at most 45")
ax.plot(np.arange(n), oval_profile, label="the profile (two passes, twice round)")
ax.set_xlabel("waypoint index")
ax.set_ylabel("speed (m/s)")
ax.set_title("oval: the cap and the forward-backward profile")
ax.legend()
plt.show()

print(f"{n} waypoints; largest difference from the agent's profile {largest_difference:.1e} m/s")
print(f"changed by the second lap: {changed_by_second_lap} waypoints")
print(f"slowest: waypoint {slowest_index} at {slowest_speed:.4f} m/s")
print(f"planned lap {planned_lap_time:.3f} s; at the cap on {fraction_at_cap:.1%} of the waypoints")

# independent checks: the agent's profile, the profile's two rules, and the slowest point is a cap
assert largest_difference < 1e-9
assert np.all(oval_profile <= cap + 1e-12)
assert np.all(oval_profile ** 2 <= oval_profile[after] ** 2 + 2.0 * 5.0 * ds_oval + 1e-9)
assert np.all(oval_profile[after] ** 2 <= oval_profile ** 2 + 2.0 * 4.0 * ds_oval + 1e-9)
assert abs(slowest_speed - float(np.min(cap))) < 1e-12
assert slowest_index == int(np.argmin(cap))
assert np.max(np.abs(lap_profile(xs, ys, 4.6, 5.0, 4.0, laps=3) - oval_profile)) == 0.0
assert 0.0 < fraction_at_cap < 1.0

## Exercise 3 - A lead in seconds

**Prediction.** max(0.3·v, 2) gives 2, 2, 3, 6, 9 and 13.5 m.
`test-circuit` starts on a straight with waypoints about 2 m apart, so the walk stops after ⌈lead / spacing⌉ segments; the segments there are just under 2 m, so every lead that is a whole multiple of 2 m needs one more segment than it seems: waypoints 2, 2, 2, 4, 5 and 7.
The lesson's §1 says the lap's last corner is just before the start line and the car is still accelerating out of it there, so the goals should be below 45 m/s and rise with the lead: a longer lead reads the profile further up the forward pass's ramp.
At 45 m/s the first two segments (about 4 m) take about 0.09 s.

In [ ]:
def lead_index(segment_length, current_index, speed, lead_s=0.3, lead_min=2.0):
    n = len(segment_length)
    lead = max(lead_s * speed, lead_min)
    j, ahead = current_index, 0.0
    while ahead < lead:
        ahead += segment_length[j % n]
        j += 1
    return j % n


track = load_track("test-circuit")
agent = Unit07(track.waypoints, None)
SPEEDS = (0.0, 5.0, 10.0, 20.0, 30.0, 45.0)
lead_metres = {speed: max(0.3 * speed, 2.0) for speed in SPEEDS}
lead_waypoints = {speed: lead_index(agent.segment_length, 0, speed) for speed in SPEEDS}
goal_speed = {speed: agent.profile[lead_waypoints[speed]] for speed in SPEEDS}
two_waypoint_lead_s = (agent.segment_length[0] + agent.segment_length[1]) / 45.0
why_seconds = (
    "The speed loop's lag is a time, so a lead in seconds gives it the same time to brake at every speed "
    "(13.5 m at 45 m/s but 3 m at 10 m/s), while a fixed two waypoints is only about 0.09 s at 45 m/s, "
    "far too short, and on a track like CARLA's, whose waypoints are not 2 m apart (or evenly spaced), "
    "a lead in waypoints would be a different distance, and a different time, everywhere."
)

for speed in SPEEDS:
    print(f"v = {speed:4.0f} m/s: lead {lead_metres[speed]:4.1f} m -> waypoint {lead_waypoints[speed]}, "
          f"goal {goal_speed[speed]:.2f} m/s")
print(f"two waypoints at 45 m/s: {two_waypoint_lead_s:.4f} s")
print(why_seconds)

# independent checks: the walk stops at the first waypoint whose path distance reaches the lead
for speed in SPEEDS:
    k = lead_waypoints[speed]  # the path from waypoint 0 to waypoint k is the sum of the first k segments
    assert sum(agent.segment_length[:k]) >= lead_metres[speed] > sum(agent.segment_length[:k - 1])
assert lead_metres == {0.0: 2.0, 5.0: 2.0, 10.0: 3.0, 20.0: 6.0, 30.0: 9.0, 45.0: 13.5}
assert all(goal_speed[speed] == agent.profile[lead_waypoints[speed]] for speed in SPEEDS)
goals = [goal_speed[speed] for speed in SPEEDS]
assert goals == sorted(goals) and max(goals) < 45.0  # still on the ramp out of the last corner
assert 0.08 < two_waypoint_lead_s < 0.1

## Exercise 4 - The friction circle along the edge

**Prediction.** The sums are 5.2² + 3² = 36.04, 5² + 4² = 41, 4.8² + 4.5² = 43.29, 4.8² + 5² = 48.04 and 4.4² + 6² = 55.36 m²/s⁴: they **grow** with A_DEC, by about 19 m²/s⁴ from the first row to the last, so the edge is nowhere near one circle.
The combined rates run from 6.0 to 7.4 m/s², about 0.61 g to 0.76 g, all well under the 1 g of μ ≈ 1 in the agent's comment.

In [ ]:
edge = {3.0: 5.2, 4.0: 5.0, 4.5: 4.8, 5.0: 4.8, 6.0: 4.4}
circle_sums = {a_dec: a_lat ** 2 + a_dec ** 2 for a_dec, a_lat in edge.items()}
combined = {a_dec: math.sqrt(total) for a_dec, total in circle_sums.items()}
fraction_of_g = {a_dec: combined[a_dec] / 9.81 for a_dec in edge}
sum_range = max(circle_sums.values()) - min(circle_sums.values())
is_constant = sum_range < 1.0
why_not_constant = (
    "The planner does not model the circle: its cornering cap and its braking pass each spend their whole "
    "budget separately, so the braking pass brakes at the full A_DEC even where the car is already cornering, "
    "and what actually decides the edge is the car arriving above the profile (the speed loop lags behind the "
    "plan, more so the harder the plan brakes) and the line pure pursuit drives, which is not the waypoints' line. "
    "The edge is where these errors use up the margin, not where the planned rates touch the tyres' circle."
)

radius = combined[3.0]
quarter_x = np.linspace(0.0, radius, 200)
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(quarter_x, np.sqrt(radius ** 2 - quarter_x ** 2), color="grey",
        label=f"a circle of radius {radius:.2f} m/s^2")
ax.plot(list(edge), list(edge.values()), "o", label="the clean edge of each A_DEC row")
ax.set_aspect("equal")
ax.set_xlabel("A_DEC (m/s^2)")
ax.set_ylabel("last clean A_LAT (m/s^2)")
ax.set_title("test-circuit's clean edge against a friction circle")
ax.legend()
plt.show()

for a_dec in edge:
    print(f"A_DEC {a_dec}: A_LAT {edge[a_dec]}, sum {circle_sums[a_dec]:.2f}, "
          f"combined {combined[a_dec]:.3f} m/s^2 = {fraction_of_g[a_dec]:.3f} g")
print(f"range of the sums: {sum_range:.2f} m^2/s^4, constant: {is_constant}")
print(why_not_constant)

# independent checks: the hand sums, and every point after the first lies outside the first one's circle
assert abs(circle_sums[3.0] - 36.04) < 1e-9 and abs(circle_sums[6.0] - 55.36) < 1e-9
assert abs(sum_range - 19.32) < 1e-9 and not is_constant
assert all(combined[a] > radius for a in (4.0, 4.5, 5.0, 6.0))
assert list(circle_sums.values()) == sorted(circle_sums.values())
assert max(fraction_of_g.values()) < 0.76

## Exercise 5 - The gain over Unit 6, on the oval

**Prediction.** The lesson's table gives medians of 216.50 s (spread 0.10) for Unit 6 and 152.75 s (0.10) for Unit 7 over seeds 0–4, so three seeds should land within a tick or two of those: a gain of about 64 s, about 29% of Unit 6's lap time, hundreds of times the spread.

In [ ]:
unit_6_results = repeat_races(UNIT_06, "oval", n=3, seeds=(0, 1, 2), car_factory=lambda: carla_like(noise=True),
                              max_seconds=600.0)
unit_7_results = repeat_races(UNIT_07, "oval", n=3, seeds=(0, 1, 2), car_factory=lambda: carla_like(noise=True),
                              max_seconds=600.0)
unit_6_times = [result.total_time for result in unit_6_results.results]
unit_7_times = [result.total_time for result in unit_7_results.results]
all_clean = all(clean(result) for result in unit_6_results.results + unit_7_results.results)
medians = {"unit-06": float(np.median(unit_6_times)), "unit-07": float(np.median(unit_7_times))}
spreads = {"unit-06": max(unit_6_times) - min(unit_6_times), "unit-07": max(unit_7_times) - min(unit_7_times)}
unit_7_better = noise_aware_better(unit_7_times, unit_6_times)
gain_s = medians["unit-06"] - medians["unit-07"]
gain_fraction = gain_s / medians["unit-06"]
three_seeds = (
    f"Yes for this comparison: a gain of {gain_s:.1f} s against spreads of a tick or two is hundreds of times "
    "the noise, but a 0.3 s difference between two Unit 7 settings is about the size of the spreads the "
    "lesson measured over five and ten seeds (0.1-0.3 s), so three seeds would not separate them."
)

print("Unit 6:", unit_6_times, unit_6_results)
print("Unit 7:", unit_7_times, unit_7_results)
print(f"all clean: {all_clean}; medians {medians}; spreads {spreads}")
print(f"Unit 7 better beyond the noise: {unit_7_better}; gain {gain_s:.2f} s ({gain_fraction:.1%})")
print(three_seeds)

# independent checks: the RepeatResult's own median and spread, and the lesson's medians
assert all_clean and unit_7_better
assert medians["unit-06"] == unit_6_results.median and medians["unit-07"] == unit_7_results.median
assert abs(spreads["unit-06"] - unit_6_results.spread) < 1e-9
assert abs(spreads["unit-07"] - unit_7_results.spread) < 1e-9
assert abs(medians["unit-06"] - 216.50) <= 2 * TICK and abs(medians["unit-07"] - 152.75) <= 2 * TICK
assert gain_s > 100 * max(spreads.values()) and 0.25 < gain_fraction < 0.33

## Exercise 6 - Where the time goes

**Prediction.** The oval is two straights joined by two equal ends.
Six equal sixths of the lap from the start (on a straight) put a straight in sections 0 and 3 and the middle of each end in sections 2 and 5.
Unit 6 never goes above 20 m/s, so it loses most where Unit 7 can reach 45 m/s: in sections 0 and 3.
In the ends both agents are held to the corner's cap (Unit 7 at A_LAT 4.6, Unit 6 at its own cap), so the gain there should be smaller.

In [ ]:
def section_of_rows(telemetry, sections):
    return sections[(telemetry["progress"] + 1) % len(sections)]


sections = grip_sections(load_track("oval"))
telemetry_6 = unit_6_results.results[0].telemetry
telemetry_7 = unit_7_results.results[0].telemetry
rows_6 = section_of_rows(telemetry_6, sections)
rows_7 = section_of_rows(telemetry_7, sections)
driving_6, driving_7 = telemetry_6["held"] == 0, telemetry_7["held"] == 0
times_6 = [float(np.sum(driving_6 & (rows_6 == k))) * CONTROL_DT for k in range(6)]
times_7 = [float(np.sum(driving_7 & (rows_7 == k))) * CONTROL_DT for k in range(6)]
section_gain = [times_6[k] - times_7[k] for k in range(6)]
biggest_gain_section = int(np.argmax(section_gain))
rounded_gain = np.round(section_gain, 9)  # whole ticks: a tie must not be decided by rounding error
smallest_gain_section = int(np.argmin(rounded_gain))  # the first one if there is a tie
tied_smallest = [k for k in range(6) if rounded_gain[k] == rounded_gain[smallest_gain_section]]
peak_speed_7 = [float(np.max(telemetry_7["v"][driving_7 & (rows_7 == k)])) for k in range(6)]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(6), times_6, "o-", label="Unit 6 (seed 0)")
ax.plot(range(6), times_7, "s-", label="Unit 7 (seed 0)")
ax.set_xlabel("grip section")
ax.set_ylabel("time in the section over three laps (s)")
ax.set_title("oval: where the time goes")
ax.legend()
plt.show()

for k in range(6):
    print(f"section {k}: Unit 6 {times_6[k]:6.2f} s, Unit 7 {times_7[k]:6.2f} s, gain {section_gain[k]:6.2f} s, "
          f"Unit 7 peak {peak_speed_7[k]:5.2f} m/s")
print(f"biggest gain in section {biggest_gain_section}, smallest in section {smallest_gain_section} "
      f"(sections tied for the smallest: {tied_smallest})")
explain = (
    f"Unit 7 gains most in sections 0 and 3, the two straights, where it reaches about 45.9 m/s against Unit 6's "
    f"20 m/s, and least in sections 2 and 5 (tied), the two ends, where both agents are held near the corner's cap "
    f"(Unit 7 peaks at only {peak_speed_7[smallest_gain_section]:.1f} m/s there): the plan's speed is won on the "
    "straights and in braking late for the corners, not in the corners themselves."
)
print(explain)

# independent checks: the section times add up to each race's driving time, and the gains to the gain
assert abs(sum(times_6) - float(np.sum(driving_6)) * CONTROL_DT) < 1e-9
assert abs(sum(times_7) - float(np.sum(driving_7)) * CONTROL_DT) < 1e-9
assert abs(sum(times_7) - unit_7_results.results[0].total_time) <= 2 * TICK
assert abs(sum(section_gain) - (sum(times_6) - sum(times_7))) < 1e-9
assert all(gain > 0.0 for gain in section_gain)
assert peak_speed_7[biggest_gain_section] > peak_speed_7[smallest_gain_section]
assert biggest_gain_section == 3 and tied_smallest == [2, 5]
assert min(peak_speed_7[0], peak_speed_7[3]) > 45.0 > 30.0 > max(peak_speed_7[2], peak_speed_7[5])

## Exercise 7 - The edge on one row

**Prediction.** The lesson's grid says the A_DEC 5.0 row is clean at 4.8 (300.90 s median) and not clean at 5.0 on any seed, so seed 0 should agree: 4.8 clean at about 300.9 s, 5.0 not clean.
Then the margin rule ships 4.6, and the margin costs about 304.15 − 300.9 ≈ 3.3 s on seed 0.

In [ ]:
row = {}
for a_lat in (4.8, 5.0):
    result = run_race(variant(UNIT_07, f"a-lat-{a_lat}", A_LAT=a_lat), "test-circuit", car=carla_like(noise=True),
                      seed=0, max_seconds=400.0)
    row[a_lat] = (clean(result), result.total_time, result.collisions, max_offset(result))
clean_values = [a_lat for a_lat in row if row[a_lat][0]]
edge_on_seed_0 = max(clean_values) if clean_values else None
margin_choice = round(edge_on_seed_0 - 0.2, 1)
seconds_given_up = (304.15 if margin_choice == 4.6 else row[margin_choice][1]) - row[edge_on_seed_0][1]
one_seed_caveat = (
    f"By the unit's rule a cell is clean only if it is clean on seeds 0-4, so A_LAT {edge_on_seed_0} would have "
    "to be raced clean on seeds 1-4 as well (and the next step shown to fail on them), and because a result on "
    "an edge needs more, it should be checked on seeds it was not tuned on (5-9) before it is trusted."
)

for a_lat in row:
    print(f"A_LAT {a_lat}: clean {row[a_lat][0]}, {row[a_lat][1]:.2f} s, {row[a_lat][2]} collisions, "
          f"max |offset| {row[a_lat][3]:.2f} m")
print(f"edge on seed 0: {edge_on_seed_0}; the margin ships {margin_choice}, giving up {seconds_given_up:.2f} s")
print(one_seed_caveat)

# independent checks: the lesson's grid row
assert row[4.8][0] and not row[5.0][0]
assert edge_on_seed_0 == 4.8 and margin_choice == 4.6
assert abs(row[4.8][1] - 300.90) <= 2 * TICK
assert 2.5 < seconds_given_up < 4.0

## Exercise 8 - Challenge: a backward pass that respects the friction circle

**Prediction.** The ellipse only lowers the braking where the car is braking **into** a corner: on a straight |κ| ≈ 0 and the full 5.0 m/s² is available, and at a waypoint at its full cap v²|κ| = A_LAT, so the segment leading into it allows no braking at all.
So every braking ramp gets longer and the profile comes down earlier and lower, but only on the ramps into corners, never on the straights or the exits (the forward pass is unchanged).
The planned lap should get slower by a few seconds, and the race too, while the car, arriving slower and less pushed, should end up a little closer to the line.

In [ ]:
def circle_profile(cap, curvature, segment_length, a_lat, a_dec, a_acc, laps=2):
    n = len(cap)
    v = list(cap)
    for _ in range(laps):
        for i in range(n - 1, -1, -1):
            j = (i + 1) % n
            used = v[j] ** 2 * abs(curvature[j]) / a_lat  # the share of the grip spent cornering at j
            a_i = a_dec * math.sqrt(max(0.0, 1.0 - used ** 2))
            v[i] = min(v[i], math.sqrt(v[j] ** 2 + 2.0 * a_i * segment_length[i]))
    for _ in range(laps):
        for i in range(n):
            j = (i + 1) % n
            v[j] = min(v[j], math.sqrt(v[i] ** 2 + 2.0 * a_acc * segment_length[i]))
    return v


def cyclic_time(v, ds):
    n = len(v)
    return sum(ds[i] / ((v[i] + v[(i + 1) % n]) / 2.0) for i in range(n))


class CircleUnit07(Unit07):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.profile = circle_profile(self.cap, self.curvature, self.segment_length, 4.6, 5.0, 4.0)


circle = circle_profile(agent.cap, agent.curvature, agent.segment_length, 4.6, 5.0, 4.0)
plain = np.array(agent.profile)
lowered_count = int(np.sum(plain - np.array(circle) > 1e-9))
largest_drop = float(np.max(plain - np.array(circle)))
planned_lap_plain = cyclic_time(agent.profile, agent.segment_length)
planned_lap_circle = cyclic_time(circle, agent.segment_length)
circle_result = run_race(CircleUnit07, "test-circuit", car=carla_like(noise=True), seed=0, max_seconds=600.0)
circle_race = (clean(circle_result), circle_result.total_time, max_offset(circle_result))
worth_it = (
    f"Respecting the circle is clean in {circle_race[1]:.2f} s with a max |offset| of {circle_race[2]:.2f} m, "
    f"against the reference's 304.15 s and 1.48 m: it bought {1.48 - circle_race[2]:.2f} m more room to the "
    f"line and cost {circle_race[1] - 304.15:.2f} s, because it brakes earlier on every ramp into a corner. "
    "To win the time back I would move A_LAT (or A_DEC) up next, since the circle-aware plan arrives at the "
    "corners with margin to spare, and re-check the edge on five seeds."
)

print(f"lowered at {lowered_count} of {len(circle)} waypoints, by at most {largest_drop:.3f} m/s")
print(f"planned lap: plain {planned_lap_plain:.2f} s, circle {planned_lap_circle:.2f} s")
print(f"race, seed 0: clean {circle_race[0]}, {circle_race[1]:.2f} s, max |offset| {circle_race[2]:.2f} m")
print(worth_it)

# independent checks: with zero curvature everywhere the ellipse is the plain pass; the circle only lowers
circle_array = np.array(circle)
assert circle_profile(agent.cap, [0.0] * len(circle), agent.segment_length, 4.6, 5.0, 4.0) == agent.profile
assert np.all(circle_array <= plain + 1e-12) and lowered_count > 0
assert np.all(circle_array <= np.array(agent.cap) + 1e-12)
assert planned_lap_circle > planned_lap_plain
assert circle_race[0]

## Exercise 9 - Challenge: does the order of the sequential pass matter?

**Prediction.** A section's knob raises the speed the car carries **out** of it, so the next section in driving order starts faster and has less room; sweeping in the order 0 → 5 tests each section on top of the one before it, while 5 → 0 tests each section with the section **before** it still at 1.0, an easier entry.
So the two orders can keep different knobs; the lesson's own table (section 1 clean alone at 1.5 but not on top of section 0's 1.2) says the answer depends on what was set first.
On one lap and one seed the result is cheaper and noisier than the lesson's five-seed pass, so it should be read only as a demonstration of the effect.

In [ ]:
def sequential_pass(order):
    grip = [1.0] * 6
    races = 0
    for section in order:
        for knob in (1.1, 1.2, 1.3, 1.4, 1.5):
            trial = list(grip)
            trial[section] = knob
            source = variant(UNIT_07, f"order-{races}-" + "-".join(str(g) for g in trial), GRIP=tuple(trial))
            result = run_race(source, "test-circuit", laps=1, car=carla_like(noise=True), seed=0, max_seconds=200.0)
            races += 1
            if not clean(result):
                break
            grip[section] = knob
    return tuple(grip), races


forward_knobs, forward_races = sequential_pass((0, 1, 2, 3, 4, 5))
backward_knobs, backward_races = sequential_pass((5, 4, 3, 2, 1, 0))
same_knobs = forward_knobs == backward_knobs

print(f"order 0 -> 5: {forward_knobs} in {forward_races} races")
print(f"order 5 -> 0: {backward_knobs} in {backward_races} races")
print(f"same knobs: {same_knobs}")
why_order_matters = (
    "A section's knob raises the speed the car carries out of it, so the next section in driving order starts "
    "faster and has less room: in the order 0 -> 5 section 0 rose to 1.4 and section 1 then failed already at 1.1, "
    "while in the order 5 -> 0 section 1 rose to 1.5 with section 0 still at 1.0, and then section 0 could only "
    "go to 1.2 before it handed that aggressive section 1 a car arriving too fast. "
    "The knobs interact, so a per-section tuning is one point found by one path through the knobs, not the best "
    "tuning, and whichever order is used it must be re-checked as a whole vector on seeds it was not tuned on."
)
print(why_order_matters)

# independent checks: every kept knob is on the grid, and the two orders disagree where they meet (sections 0, 1)
assert all(knob in (1.0, 1.1, 1.2, 1.3, 1.4, 1.5) for knob in forward_knobs + backward_knobs)


def races_needed(knobs):
    # a section kept at knob k raced every step up to k, plus the failing step after it (none after 1.5)
    return sum(round((k - 1.0) / 0.1) + (0 if k == 1.5 else 1) for k in knobs)


assert forward_races == races_needed(forward_knobs) and backward_races == races_needed(backward_knobs)
assert not same_knobs
assert forward_knobs[:2] == (1.4, 1.0) and backward_knobs[:2] == (1.2, 1.5)
assert forward_knobs[2:] == backward_knobs[2:]